# Calibrate AnomalyCLIP category thresholds on MVTec train/good

This thin notebook runs the shared adversarial harness calibration entrypoint. It never uses MVTec test images or labeled anomalies. The resulting JSON can be mounted and reused by later benchmark notebooks that use the same checkpoint, image size, target configuration, and quantile.

In [ ]:
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE/UPDATE REPOSITORIES =====')
working = Path('/kaggle/working')
repo_root = working / 'adversarial-robustness'
anomalyclip_root = working / 'AnomalyCLIP'

repo_url = 'https://github.com/Parsagh05/adversarial-robustness.git'
if repo_root.exists():
    subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_root)], check=True)

official_url = 'https://github.com/zqhang/AnomalyCLIP.git'
if anomalyclip_root.exists():
    subprocess.run(['git', '-C', str(anomalyclip_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', official_url, str(anomalyclip_root)], check=True)

experiment_root = repo_root
requirements = experiment_root / 'requirements.txt'
print('===== STEP 2: INSTALL DEPENDENCIES =====')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('Environment ready.')

In [ ]:
import json
import os
import sys
from pathlib import Path
import torch

experiment_root = Path('/kaggle/working/adversarial-robustness')
if str(experiment_root) not in sys.path:
    sys.path.insert(0, str(experiment_root))

from adversarial_harness import AttackConfig, ExperimentConfig, calibrate_thresholds

print('===== STEP 3: RESOLVE DATA AND CHECKPOINT =====')
MVTEC_ROOT = Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection')
ANOMALYCLIP_ROOT = Path('/kaggle/working/AnomalyCLIP')
CHECKPOINT_PATH = ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale' / 'epoch_15.pth'
OUTPUT_ROOT = Path('/kaggle/working/anomalyclip_thresholds_q95')

if not MVTEC_ROOT.is_dir():
    raise FileNotFoundError(f'MVTec mount not found: {MVTEC_ROOT}')
if not CHECKPOINT_PATH.is_file():
    available = sorted((ANOMALYCLIP_ROOT / 'checkpoints').rglob('*.pth'))
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}. Available: {available}')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before calibration.')

os.environ['ANOMALYCLIP_ROOT'] = str(ANOMALYCLIP_ROOT)
os.environ['ANOMALYCLIP_CHECKPOINT'] = str(CHECKPOINT_PATH)

attack = AttackConfig(
    image_size=518,
    scopes=('dataset',),
    directions=('normal_to_abnormal',),
    loss_modes=('global',),
)
config = ExperimentConfig(
    mvtec_root=str(MVTEC_ROOT),
    output_root=str(OUTPUT_ROOT),
    anomalyclip_root=str(ANOMALYCLIP_ROOT),
    anomalyclip_checkpoint=str(CHECKPOINT_PATH),
    device='cuda',
    target_model='AnomalyCLIP',
    categories=None,
    target_batch_size=2,
    compute_lpips=False,
    threshold_mode='normal_train_quantile',
    threshold_quantile=0.95,
    thresholds_path=None,
    resume=True,
    attack=attack,
)

print('===== STEP 4: CALIBRATE ON TRAIN/GOOD ONLY =====')
threshold_path = calibrate_thresholds(config)
payload = json.loads(threshold_path.read_text(encoding='utf-8'))
print('category | threshold | count | min | mean | max | std')
for category, record in payload['categories'].items():
    print(
        f"{category:12s} | {record['threshold']:.6f} | "
        f"{record['sample_count']:5d} | {record['score_min']:.6f} | "
        f"{record['score_mean']:.6f} | {record['score_max']:.6f} | "
        f"{record['score_std']:.6f}"
    )
print('Threshold artifact:', threshold_path)
print('Raw calibration scores:', OUTPUT_ROOT / 'normal_train_scores.npz')

In [ ]:
import shutil
from pathlib import Path

archive = shutil.make_archive(
    '/kaggle/working/anomalyclip_thresholds_q95',
    'zip',
    root_dir=Path('/kaggle/working/anomalyclip_thresholds_q95'),
)
print('Packaged calibration artifacts:', archive)